# Assignment 4: Semantic Search on Vision 2030 PDF

**Goal:** Build a semantic search pipeline:
1. **Load** the Vision 2030 PDF
2. **Split** it into chunks
3. **Embed** chunks using HuggingFace sentence-transformers
4. **Store** in an in-memory vector store
5. **Retrieve** relevant chunks for natural language queries

## 1. Load the PDF

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = os.path.join(os.getcwd(), "vision2030.pdf")
loader = PyPDFLoader(pdf_path)
docs = loader.load()

print(f"Loaded {len(docs)} pages from Vision 2030 PDF")
print(f"\n-- First page preview (300 chars) --")
print(docs[0].page_content[:300])
print(f"\n-- Metadata --")
print(docs[0].metadata)

ModuleNotFoundError: No module named 'langchain_huggingface'

## 2. Split & Embed

In [2]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(f"Split into {len(all_splits)} chunks")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

sample_vector = embeddings.embed_query(all_splits[0].page_content)
print(f"Embedding dimension: {len(sample_vector)}")
print(f"   First 5 values: {sample_vector[:5]}")

NameError: name 'RecursiveCharacterTextSplitter' is not defined

## 3. Store in Vector Store

In [3]:
vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)
print(f"Stored {len(ids)} chunks in the vector store")

NameError: name 'embeddings' is not defined

## 4. Semantic Search Results

In [4]:
queries = [
    "What is the main goal of Saudi Vision 2030?",
    "How will the economy be diversified away from oil?",
    "What are the plans for tourism in Saudi Arabia?",
    "How is education being reformed?",
    "What is the role of the private sector?",
]

print("=" * 70)
print("SEMANTIC SEARCH RESULTS")
print("=" * 70)

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 50)

    results = vector_store.similarity_search(query, k=2)
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] (page: {doc.metadata.get('page', '?')})")
        print(f"      {doc.page_content[:150]}...")

    scored_results = vector_store.similarity_search_with_score(query, k=1)
    doc, score = scored_results[0]
    print(f"  Best match score: {score:.4f}")

SEMANTIC SEARCH RESULTS

Query: What is the main goal of Saudi Vision 2030?
--------------------------------------------------


NameError: name 'vector_store' is not defined

## 5. Retriever Batch Query

In [5]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

batch_results = retriever.batch([
    "What are the healthcare goals?",
    "How will housing be improved?",
])

print("=" * 70)
print("RETRIEVER BATCH RESULTS")
print("=" * 70)

for i, result_docs in enumerate(batch_results):
    print(f"\n  Query {i+1} -> page: {result_docs[0].metadata.get('page', '?')}")
    print(f"    {result_docs[0].page_content[:150]}...")

NameError: name 'vector_store' is not defined